In [11]:
from langchain_community.document_loaders import BSHTMLLoader
from langchain_ollama import OllamaEmbeddings


import bs4
from langchain import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from bs4 import BeautifulSoup
from langchain_experimental.text_splitter import SemanticChunker
import os
import re

In [12]:
def read_documents(directory):
    # Read documents from the specified directory
    documents = dict()
    for filename in os.listdir(directory):
        new_file_path = os.path.join(directory, filename)

        if os.path.isdir(new_file_path):
            for filename in os.listdir(new_file_path):
                with open(os.path.join(new_file_path, filename), "r") as file:
                    documents[os.path.basename(new_file_path)] = file.read()
        else:
            with open(os.path.join(directory, filename), "r") as file:
                
                documents[filename] = file.read()
    return documents

In [13]:
docs = read_documents('../filings')

In [ ]:
def file_parsing_custom(doc_dict, ticker):
    soup = BeautifulSoup(doc_dict[ticker], 'html.parser')
    embeddings_model = OllamaEmbeddings(model="all-minilm")
    text_list = []
    for div in soup.find_all('div'):
        if not div.find('table'):
            text_list.append(div.get_text(separator="\n",strip=True))
    text_list= [line for text in text_list for line in text.split('\n')]
    text_list = [re.sub(r'\xa0', ' ', text) for text in text_list[1:]]
    seen = set()
    to_docs = []

    for text in text_list[1:]:
        if text and text not in seen:
            text = text.lower()
            seen.add(text)
            to_docs.append(text)
    heads = [tod for tod in to_docs if re.match(r"item .*\.", tod) ]
    section = None 
    doc_list = []
    for tod in to_docs:
        if tod in heads:
            #create a string that combines the section text and then create a document with the full section text 
            if section == None:
                section = tod
                section_text = ""
            else:
                text_splitter = SemanticChunker(embeddings_model, breakpoint_threshold_type="gradient", breakpoint_threshold_amount = 92.5)
                docs = text_splitter.create_documents([section_text], metadatas=[{"ticker":ticker, "section":section}])
                # doc = Document(page_content=section_text, metadata={"ticker": ticker, "section": section})
                # doc_list.append(doc)
                doc_list.extend(docs)
                section = tod
                section_text = ""
        elif section:
            section_text += tod

    return doc_list

In [ ]:
def file_parsing_semantic(doc_dict, ticker, embed_model):
    soup = BeautifulSoup(doc_dict[ticker], 'html.parser')
    embeddings_model = OllamaEmbeddings(model=embed_model)
    text_list = []
    for div in soup.find_all('div'):
        if not div.find('table'):
            text_list.append(div.get_text(separator="\n", strip=True))


    seen = set()
    result = ''
    for text in text_list[1:]:
        if text and len(text) > 20 and text not in seen:
            seen.add(text)
            result += text

    text_splitter = SemanticChunker(embeddings_model, buffer_size=3,breakpoint_threshold_type="gradient", breakpoint_threshold_amount = 90)
    docs = text_splitter.create_documents([result], metadatas = [{"ticker":ticker}])

    return docs

In [ ]:
parsed_html =[]

for ticker in list(docs.keys()):
    parsed_html.extend(file_parsing_semantic(docs,ticker,"all-minilm"))

In [18]:
from langchain_chroma import Chroma
embeddings_model = OllamaEmbeddings(model="all-minilm")
vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings_model,
    persist_directory="./chroma_semantic_chunking"
)

In [19]:
batch_size = 5461  

for i in range(0, len(parsed_html), batch_size):
    chunk = parsed_html[i:i + batch_size]
    vector_store.add_documents(chunk)

In [ ]:
data = vector_store._collection.get()

